# **6. DQN y PPO para _cartpole_ con Stable-Baselines3**

En [03_cartpole-dqn.ipynb](../from-scratch/03_cartpole-dqn.ipynb) y [04_cartpole-reinforce.ipynb](../from-scratch/04_cartpole-reinforce.ipynb) implementamos nosotros mismos DQN y REINFORCE. En este cuaderno, resolvemos el mismo problema con [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3), una librería de implementaciones fiables y probadas de los algoritmos de aprendizaje por refuerzo profundo (DRL) más comunes, construida sobre PyTorch. Con SB3, el agente, la red neuronal, la memoria de experiencias (_replay buffer_) y el bucle de entrenamiento ya están implementados: solo tenemos que elegir un algoritmo, fijar sus hiperparámetros y llamar a `learn()`.

Entrenaremos dos agentes:

*   [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html): el mismo algoritmo basado en valor de [03_cartpole-dqn.ipynb](../from-scratch/03_cartpole-dqn.ipynb).
*   [PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html): un algoritmo de gradiente de política (actor-crítico) que mejora las ideas de REINFORCE ([04_cartpole-reinforce.ipynb](../from-scratch/04_cartpole-reinforce.ipynb)).

Basado en:
> A. Raffin et al. (2021) Stable-Baselines3: Reliable Reinforcement Learning Implementations. _Journal of Machine Learning Research_ 22(268):1-8. Disponible en [GitHub](https://github.com/DLR-RM/stable-baselines3).

> A. Raffin (2020) RL Baselines3 Zoo. Disponible en [GitHub](https://github.com/DLR-RM/rl-baselines3-zoo).

Comenzamos importando las librerías necesarias, así como otras utilidades que emplearemos más adelante:

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

import io
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
from PIL import Image

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def show_episode(env, act):
    """
    Ejecuta un episodio completo y lo muestra como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción a realizar en un estado
    """
    state, info = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    terminated = truncated = False

    # Bucle de interacción agente-entorno (1 episodio)
    while not (terminated or truncated):
        action = act(state)
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(Image.fromarray(env.render()))
        score += reward

    # Guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format="gif", save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata["render_fps"], loop=0)
    display(IPImage(data=gif.getvalue(), format="gif"))
    print("Puntuación: ", score)

def monitor_scores(env):
    """
    Devuelve la puntuación de cada episodio registrada por Monitor, en el orden en que terminaron.

    Parámetros
    ==========
        env: entorno (o entorno vectorizado) envuelto con Monitor
    """
    # Un entorno vectorizado contiene varias copias del entorno envueltas con Monitor
    monitors = [env] if isinstance(env, Monitor) else env.envs
    scores = np.concatenate([m.get_episode_rewards() for m in monitors])
    ends = np.concatenate([np.cumsum(m.get_episode_lengths()) for m in monitors])
    return scores[np.argsort(ends)]   # Ordenar los episodios por el paso de tiempo (de cada entorno) en que terminaron

def plot_scores(scores, title, solved_score=None, window=100):
    """
    Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label="Puntuación")
    plt.plot(np.arange(len(scores)), avg, label=f"Puntuación media (últimos {window})")
    if solved_score is not None:
        plt.axhline(solved_score, color="gray", linestyle="--", label="Resuelto")
    plt.ylabel("Puntuación")
    plt.xlabel("Episodio nº")
    plt.title(title)
    plt.legend()
    plt.show()

## **6.1. Entorno**

Creamos el entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/). El estado es un vector de $4$ valores continuos (posición y velocidad del carro, ángulo y velocidad angular del péndulo) y hay $2$ acciones posibles: `0` (empujar el carro a la izquierda) y `1` (empujarlo a la derecha).

El entorno se considera resuelto cuando el agente obtiene una puntuación media de al menos $475$ puntos en 100 episodios consecutivos. Los episodios se truncan a los $500$ pasos, que es por tanto la puntuación máxima.

In [ ]:
SOLVED_SCORE = 475.0   # Puntuación media (últimos 100 episodios) para considerar resuelto CartPole-v1

env = gym.make("CartPole-v1", render_mode="rgb_array")

print(env.observation_space)
print(env.action_space)

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

La función `show_episode()` muestra un episodio como una animación, junto con su **puntuación**, es decir, la suma de las recompensas obtenidas a lo largo del episodio. Veamos, por ejemplo, un episodio de un agente que elige acciones al azar:

In [ ]:
show_episode(env, lambda state: env.action_space.sample())

## **6.2. Algoritmo DQN**

La clase [`DQN`](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) incluye la red Q, la red objetivo (_target network_), la memoria de experiencias (_replay buffer_) y la estrategia de exploración $\epsilon$-_greedy_. Comparemos sus parámetros con los de nuestra implementación en [dqn_agent.py](../from-scratch/dqn_agent.py):

| Parámetro de SB3 | Significado | `dqn_agent.py` |
|---|---|---|
| `buffer_size` | tamaño de la memoria de experiencias | `BUFFER_SIZE` |
| `batch_size` | tamaño del minilote | `BATCH_SIZE` |
| `gamma` | factor de descuento | `GAMMA` |
| `learning_rate` | tasa de aprendizaje | `LR` |
| `train_freq` | cada cuántos pasos se actualiza la red | (en cada paso) |
| `target_update_interval` | cada cuántos pasos se actualiza la red objetivo | `UPDATE_EVERY`, `TAU` (actualización suave) |
| `exploration_*` | calendario de $\epsilon$-_greedy_ | `eps_start`, `eps_end`, `eps_decay` |
| `policy_kwargs` | arquitectura de la red Q | [model.py](../from-scratch/model.py) |

Usamos los hiperparámetros ajustados para CartPole-v1 en el [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml). En lugar de un número de episodios, SB3 entrena durante un número dado de pasos de tiempo (`total_timesteps`).

SB3 necesita que el entorno de entrenamiento esté envuelto en un [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) para registrar la puntuación de cada episodio. Usaremos estos valores para representar las curvas de aprendizaje.

In [ ]:
env_dqn = Monitor(gym.make("CartPole-v1"))

model_dqn = DQN(
    "MlpPolicy",                        # Red Q: perceptrón multicapa
    env_dqn,
    learning_rate=2.3e-3,
    batch_size=64,
    buffer_size=100_000,
    learning_starts=1000,               # Pasos con acciones aleatorias antes de empezar a aprender
    gamma=0.99,
    target_update_interval=10,
    train_freq=256,                     # Actualizar la red cada 256 pasos...
    gradient_steps=128,                 # ...con 128 minilotes
    exploration_fraction=0.16,          # Epsilon decrece linealmente durante el primer 16% del entrenamiento...
    exploration_final_eps=0.04,         # ...hasta este valor
    policy_kwargs=dict(net_arch=[256, 256]),
    seed=0,
    verbose=0,
)

model_dqn.learn(total_timesteps=50_000, progress_bar=True)
model_dqn.save("cartpole-sb3-dqn")   # Guardar el agente entrenado

plot_scores(monitor_scores(env_dqn), "DQN", SOLVED_SCORE)

La puntuación mostrada durante el entrenamiento se obtiene con la política $\epsilon$-_greedy_, es decir, con algunas acciones aleatorias. Para evaluar la política aprendida, evaluamos el agente con la política _greedy_ (`deterministic=True`) durante 100 episodios.

In [ ]:
mean_score, std_score = evaluate_policy(model_dqn, Monitor(gym.make("CartPole-v1")), n_eval_episodes=100, deterministic=True)
print(f"DQN\tPuntuación media (100 episodios): {mean_score:.2f} +/- {std_score:.2f}")

## **6.3. Algoritmo PPO**

[Proximal Policy Optimization](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html) (PPO) es un método de gradiente de política, como REINFORCE, con varias mejoras:

*   Es un método _actor-critic_ (actor-crítico): además de la red de política (actor), aprende una red de valor (crítico) que se usa para reducir la varianza de la estimación del gradiente (en lugar del retorno normalizado $G_t$ que usamos en REINFORCE).
*   Limita (_clipping_) el tamaño de cada actualización de la política, de modo que la nueva política no se aleje demasiado de la anterior. Esto permite reutilizar cada lote de experiencia para varios pasos de gradiente (`n_epochs`).
*   Recoge experiencia de varias copias del entorno en paralelo (`n_envs`), que SB3 construye con [`make_vec_env`](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html).

De nuevo usamos los hiperparámetros del [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/ppo.yml). Nótese que solo necesitamos cambiar la clase del modelo: el resto del código es el mismo.

In [ ]:
env_ppo = make_vec_env("CartPole-v1", n_envs=8, seed=0)   # 8 entornos en paralelo, cada uno envuelto con Monitor

model_ppo = PPO(
    "MlpPolicy",                        # Redes de actor y crítico: perceptrones multicapa
    env_ppo,
    learning_rate=1e-3,
    n_steps=32,                         # Pasos recogidos de cada entorno antes de cada actualización
    batch_size=256,
    n_epochs=20,                        # Pasos de gradiente (épocas) con cada lote de experiencia
    gamma=0.98,
    gae_lambda=0.8,
    clip_range=0.2,                     # Cambio máximo de la política en cada actualización
    ent_coef=0.0,
    seed=0,
    verbose=0,
)

model_ppo.learn(total_timesteps=100_000, progress_bar=True)
model_ppo.save("cartpole-sb3-ppo")   # Guardar el agente entrenado

plot_scores(monitor_scores(env_ppo), "PPO", SOLVED_SCORE)

In [ ]:
mean_score, std_score = evaluate_policy(model_ppo, Monitor(gym.make("CartPole-v1")), n_eval_episodes=100, deterministic=True)
print(f"PPO\tPuntuación media (100 episodios): {mean_score:.2f} +/- {std_score:.2f}")

## **6.4. Visualizar el agente entrenado**

Cargamos los agentes entrenados y ejecutamos cada uno durante un episodio con la política _greedy_ (`deterministic=True`):

In [ ]:
# Cargar los agentes entrenados desde `cartpole-sb3-dqn.zip` y `cartpole-sb3-ppo.zip`
for name, model_load in [("DQN", DQN.load("cartpole-sb3-dqn")), ("PPO", PPO.load("cartpole-sb3-ppo"))]:
    print(name)
    show_episode(env, lambda state: int(model_load.predict(state, deterministic=True)[0]))

¡Los dos agentes mantienen el péndulo en equilibrio durante todo el episodio!

En este cuaderno, hemos resuelto CartPole con dos algoritmos de Stable-Baselines3: DQN, basado en valor, y PPO, de gradiente de política. Cambiar de algoritmo solo requiere cambiar la clase del modelo y sus hiperparámetros.